# Train on a Colab GPU

In Colab, select **Runtime > Change runtime type > GPU** before running these cells. This project currently supports Python 3.11 and 3.12.

In [ ]:
import sys

if not (3, 11) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError(f"This setup expects Python 3.11 or 3.12; Colab has {sys.version.split()[0]}.")

print(f"Python {sys.version.split()[0]} is within the supported project range.")

## Get the repository

Colab runs in its own VM, so clone the project there to make its source, configs, and dataset artifacts available.

In [ ]:
import os
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/llm-internals")
REPO_URL = "https://github.com/KrishnaTarun/llm-internals.git"

if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print(f"Project directory: {Path.cwd()}")

## Install dependencies with uv

The platform markers in `pyproject.toml` select the Linux/CUDA 12.6 PyTorch dependencies in Colab. `uv sync --locked` installs the project using the versions recorded in `uv.lock`, into the repository's `.venv`. The next cell uses pip only to bootstrap the `uv` command; `uv` installs the project dependencies.

In [ ]:
%pip install uv

In [ ]:
!uv sync --locked

## Verify CUDA

This checks CUDA from the project's `uv` environment, not from Colab's separate notebook-kernel environment.

In [ ]:
!uv run python -c "import torch; print(f'PyTorch: {torch.__version__}'); print(f'CUDA runtime: {torch.version.cuda}'); assert torch.cuda.is_available(), 'CUDA is unavailable: enable a GPU runtime and restart Colab'; print(f'GPU: {torch.cuda.get_device_name(0)}')"

## Train GPT-style

The supplied config points to `output/checkpoints/GPTstyle/model.pt`, so it resumes from that checkpoint if present. Set `model.resume_from: null` in `configs/gptstyle_training.yaml` to start fresh. Colab storage is temporary; copy the checkpoint to Drive if you need to keep it after the session.

In [ ]:
!uv run python src/train.py --config configs/gptstyle_training.yaml